# drone-burst-sr sur Google Colab (GPU gratuit)

Ce notebook installe **drone-burst-sr** avec l'algorithme *Handheld Multi-Frame Super-Resolution*
(implémentation IPOL, Numba + CUDA) et lance l'interface web sur un GPU NVIDIA de Colab.

**Avant de commencer** : menu *Exécution → Modifier le type d'exécution → GPU T4*, puis exécutez
les cellules dans l'ordre.

In [ ]:
!nvidia-smi

## 1. Installation

Si le dépôt GitHub est **privé**, ajoutez un jeton GitHub (lecture seule) dans les *Secrets* de
Colab (icône 🔑 à gauche) sous le nom `GITHUB_TOKEN`. Pour un dépôt public, rien à faire.

In [ ]:
REPO = "sylvaingranges-pixel/superpixel"   # remplacez par <compte>/drone-burst-sr si le dépôt est renommé
BRANCH = "ccr-cc33588e-5l482s"             # ou "main"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
auth = f"{token}@" if token else ""
url = f"git+https://{auth}github.com/{REPO}@{BRANCH}"
!pip install -q "drone-burst-sr[gpu] @ {url}"

### Compatibilité CUDA

L'algorithme amont installe la variante **CUDA 13** de Numba, qui demande un pilote NVIDIA récent
(≥ 580). Si le pilote de Colab est plus ancien (`CUDA Version: 12.x` dans `nvidia-smi`), la cellule
suivante bascule automatiquement sur CUDA 12.

In [ ]:
import re, subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*(\d+)", out)
cuda_major = int(m.group(1)) if m else 0
print("CUDA du pilote :", cuda_major)
if cuda_major and cuda_major < 13:
    !pip install -q "numba-cuda[cu12]" "cuda-bindings>=12.9,<13" "cuda-toolkit[cccl,cudart,nvcc,nvrtc]==12.*" "nvidia-nvjitlink-cu12"
    print("Variante CUDA 12 installée.")

In [ ]:
from drone_burst_sr.gpu import detect_gpu
print(detect_gpu().describe())

## 2. Auto-test (≈ 2 min, dont ~1 min de compilation des kernels CUDA)

Rafale synthétique à décalages sous-pixel connus : vérifie que la fusion ×2 est plus fidèle et
plus nette qu'une seule image dématricée par rawpy.

In [ ]:
!python -m drone_burst_sr.selftest

## 3a. Interface web

Ouvrez le lien public `https://….gradio.live` affiché ci-dessous (valable 72 h) dans votre
navigateur Windows. Déposez les DNG, ou indiquez un dossier Google Drive (voir 3b).
La cellule reste « en cours » tant que l'interface tourne : interrompez-la pour l'arrêter.

> Les fichiers transitent par le tunnel Gradio. Pour de grosses rafales (48 MP ≈ 1 Go), il est
> plus rapide de les déposer sur Google Drive et d'indiquer le dossier dans l'interface.

In [ ]:
from drone_burst_sr.app import build_app
demo = build_app()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=True)

## 3b. Ligne de commande avec Google Drive

Copiez la rafale dans un dossier de votre Drive (par ex. `MyDrive/rafales/rafale1`).
Les résultats sont écrits à côté, dans le Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
INPUT = "/content/drive/MyDrive/rafales/rafale1"
OUT = INPUT + "_fusion_x2.tif"
!python -m drone_burst_sr --input "{INPUT}" --scale 2 --out "{OUT}" --single-out "{INPUT}_image_seule_x2.tif"

In [ ]:
# Téléchargement direct (sinon récupérez les fichiers dans Google Drive)
from google.colab import files
files.download(OUT.replace(".tif", ".jpg"))